In [1]:
import re
import pandas as pd
import numpy as np
from scipy import stats
from google.colab import files

In [2]:
up = files.upload()
nama_file = list(up.keys())[0]
print("File terbaca:", nama_file)

Saving Respons Topik 2 – Kebiasaan Digital – 3 TI A (1).xlsx to Respons Topik 2 – Kebiasaan Digital – 3 TI A (1).xlsx
File terbaca: Respons Topik 2 – Kebiasaan Digital – 3 TI A (1).xlsx


In [3]:
JUMLAH_KOLOM_DEMOGRAFI = 5

df = pd.read_excel(nama_file)
butir = df.iloc[:, JUMLAH_KOLOM_DEMOGRAFI:]

print("Jumlah responden :", len(df))
print("Jumlah butir     :", butir.shape[1])
print("Data kosong      :", int(butir.isna().sum().sum()))

Jumlah responden : 39
Jumlah butir     : 12
Data kosong      : 0


In [4]:
skala = {
    "Sangat Tidak Setuju (STS)": 1,
    "Tidak Setuju (TS)": 2,
    "Netral (N)": 3,
    "Setuju (S)": 4,
    "Sangat Setuju (SS)": 5,
}

num = butir.replace(skala).apply(pd.to_numeric, errors="coerce")

# Harus 0. Jika lebih dari 0, ada jawaban yang tidak terbaca.
print("Sel yang gagal dikonversi:", int(num.isna().sum().sum()))
num.head()

Sel yang gagal dikonversi: 0


/tmp/ipykernel_500/3609573047.py:9: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  num = butir.replace(skala).apply(pd.to_numeric, errors="coerce")


,Kebiasaan penggunaan teknologi digital [TD4. Saya membuka konten hiburan sambil menggunakan AI untuk mengerjakan tugas kuliah.],Kebiasaan penggunaan teknologi digital [TD5. Saya beralih ke aplikasi lain sebelum selesai membaca jawaban AI untuk tugas kuliah.],Kebiasaan penggunaan teknologi digital [TD6. Saya menggunakan jawaban AI dalam tugas kuliah setelah membacanya sekilas.],Prokrastinasi akademik [PA1. Saya menunda memulai tugas kuliah hingga mendekati batas pengumpulan.],Prokrastinasi akademik [PA2. Saya memulai pengerjaan tugas kuliah lebih lambat daripada waktu yang saya rencanakan.],Prokrastinasi akademik [PA3. Saya menunda melanjutkan tugas kuliah yang sudah saya mulai.],Prokrastinasi akademik [PA4. Saya menyelesaikan tugas kuliah pada saat waktu pengumpulan hampir berakhir.],Respons emosional [RE1. Saya merasa tertekan ketika waktu pengerjaan tugas menjadi singkat akibat menunda.],Respons emosional [RE2. Saya merasa cemas ketika tugas yang saya tunda mendekati batas pengumpulan.],Respons emosional [RE3. Saya merasa panik ketika beberapa tugas yang saya tunda harus segera dikumpulkan.],Respons emosional [RE4. Saya merasa gelisah ketika memikirkan tugas yang pengerjaannya saya tunda.],Respons emosional [RE5. Saya merasa kewalahan menghadapi tugas yang menumpuk akibat penundaan.]
0,3,4,2,1,5,2,4,1,4,1,2,2
1,3,2,3,3,2,3,2,4,3,4,3,4
2,4,3,5,2,4,5,1,4,5,2,1,5
3,1,1,2,1,2,1,1,5,4,5,4,5
4,4,5,4,4,5,4,5,5,4,5,3,4


In [5]:
dimensi = {}
for kolom in butir.columns:
    nama = re.match(r"^(.*?) \[", kolom).group(1)
    dimensi.setdefault(nama, []).append(kolom)

for nama, daftar in dimensi.items():
    print(f"{nama}: {len(daftar)} butir")

Kebiasaan penggunaan teknologi digital: 3 butir
Prokrastinasi akademik: 4 butir
Respons emosional: 5 butir


In [6]:
n = len(num)
t = stats.t.ppf(0.975, n - 2)
r_tabel = t / np.sqrt((n - 2) + t**2)
print(f"n = {n} | r tabel (α=0,05; df={n-2}) = {r_tabel:.3f}")

n = 39 | r tabel (α=0,05; df=37) = 0.316


In [7]:
hasil_validitas = []
for nama, daftar in dimensi.items():
    x = num[daftar]
    total = x.sum(axis=1)
    for i, kolom in enumerate(daftar, 1):
        r, p = stats.pearsonr(x[kolom], total)
        r_koreksi, _ = stats.pearsonr(x[kolom], total - x[kolom])
        hasil_validitas.append({
            "Dimensi": nama,
            "Butir": f"B{i}",
            "r hitung": round(r, 3),
            "Corrected r": round(r_koreksi, 3),
            "p-value": round(p, 4),
            "Keterangan": "Valid" if r > r_tabel else "Tidak valid",
        })

tabel_validitas = pd.DataFrame(hasil_validitas)
tabel_validitas

,Dimensi,Butir,r hitung,Corrected r,p-value,Keterangan
0,Kebiasaan penggunaan teknologi digital,B1,0.772,0.341,0.0000,Valid
1,Kebiasaan penggunaan teknologi digital,B2,0.539,0.038,0.0004,Valid
2,Kebiasaan penggunaan teknologi digital,B3,0.609,0.126,0.0000,Valid
3,Prokrastinasi akademik,B1,0.490,0.079,0.0015,Valid
4,Prokrastinasi akademik,B2,0.609,0.279,0.0000,Valid
5,Prokrastinasi akademik,B3,0.637,0.289,0.0000,Valid
6,Prokrastinasi akademik,B4,0.606,0.127,0.0000,Valid
7,Respons emosional,B1,0.571,0.144,0.0001,Valid
8,Respons emosional,B2,0.375,0.060,0.0187,Valid
9,Respons emosional,B3,0.681,0.396,0.0000,Valid


In [8]:
def cronbach_alpha(x):
    k = x.shape[1]
    return k / (k - 1) * (1 - x.var(ddof=1).sum() / x.sum(axis=1).var(ddof=1))

def kategori(a):
    return ("Sangat baik" if a >= 0.90 else "Baik" if a >= 0.80
            else "Reliabel" if a >= 0.70 else "Tidak reliabel")

hasil_reliabilitas = []
alpha_dihapus = []

for nama, daftar in dimensi.items():
    x = num[daftar]
    a = cronbach_alpha(x)
    hasil_reliabilitas.append({
        "Dimensi": nama,
        "Jumlah butir": len(daftar),
        "Cronbach's alpha": round(a, 3),
        "Kategori": kategori(a),
    })
    for i, kolom in enumerate(daftar, 1):
        if len(daftar) > 2:
            alpha_dihapus.append({
                "Dimensi": nama,
                "Butir": f"B{i}",
                "Alpha jika dihapus": round(cronbach_alpha(x.drop(columns=kolom)), 3),
            })

tabel_reliabilitas = pd.DataFrame(hasil_reliabilitas)
tabel_alpha_dihapus = pd.DataFrame(alpha_dihapus)
display(tabel_reliabilitas)
display(tabel_alpha_dihapus)

,Dimensi,Jumlah butir,Cronbach's alpha,Kategori
0,Kebiasaan penggunaan teknologi digital,3,0.288,Tidak reliabel
1,Prokrastinasi akademik,4,0.349,Tidak reliabel
2,Respons emosional,5,0.311,Tidak reliabel


,Dimensi,Butir,Alpha jika dihapus
0,Kebiasaan penggunaan teknologi digital,B1,-0.251
1,Kebiasaan penggunaan teknologi digital,B2,0.450
2,Kebiasaan penggunaan teknologi digital,B3,0.280
3,Prokrastinasi akademik,B1,0.400
4,Prokrastinasi akademik,B2,0.192
5,Prokrastinasi akademik,B3,0.170
6,Prokrastinasi akademik,B4,0.370
7,Respons emosional,B1,0.274
8,Respons emosional,B2,0.330
9,Respons emosional,B3,0.038


In [9]:
gabung = tabel_validitas.merge(tabel_alpha_dihapus, on=["Dimensi", "Butir"], how="left")

with pd.ExcelWriter("hasil_uji_validitas_reliabilitas.xlsx") as w:
    gabung.to_excel(w, sheet_name="Validitas", index=False)
    tabel_reliabilitas.to_excel(w, sheet_name="Reliabilitas", index=False)

files.download("hasil_uji_validitas_reliabilitas.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>